```markdown
Bu kod örneği, modern RAG (Retrieval Augmented Generation) sistemlerinde çok önemli bir teknik olan hibrit arama implementasyonunu gösteriyor.
Ana özellikler:

Çifte güçlü arama: Hem semantik anlam (vektörler) hem de exact kelime eşleştirmesi (BM25)
LangChain entegrasyonu: Kolayca genişletilebilir ve başka sistemlerle entegre edilebilir
Pinecone'un serverless altyapısı: Otomatik ölçeklendirme ve düşük maliyet

Dikkat edilmesi gerekenler:

API anahtarının güvenli saklanması kritik
BM25 modelinin doğru eğitilmesi sonuç kalitesini çok etkiler
Vektör boyutu (384) ve metrik seçimi (dotproduct) embedding modeliyle uyumlu olmalı

Bu tür sistemler özellikle çok dilli arama, teknik dokuman arama ve e-ticaret gibi alanlarda çok başarılı sonuçlar veriyor.
```

```markdown
1. Kurulum ve Başlangıç
Pinecone vektör veritabanı istemcisini yükler
%pip Jupyter notebook'ta paket yüklemek için kullanılır
```

In [33]:
%pip install pinecone-client

  Using cached pinecone_client-6.0.0-py3-none-any.whl.metadata (3.4 kB)
Using cached pinecone_client-6.0.0-py3-none-any.whl (6.7 kB)
Note: you may need to restart the kernel to use updated packages.


```markdown
Adım 2: API Key Tanımlaması
```

In [34]:
api_key = "pcsk_2VCtK8_4mYR34ex6bqSUXyF1VBRRY6AzxKeG2Y4HXiAZJDu7MuS1pU7g918FF9Js34gccg"

```markdown
Adım 3: LangChain Retriever İmportu
LangChain'in Pinecone hibrit arama modülünü import eder
Bu modül hem vektör hem de sparse (seyrek) arama yapabilir
```

In [35]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

```markdown
Adım 4: Pinecone İndeks Kurulumu
başlatılır
İndeks kontrolü: Belirtilen isimde indeks var mı kontrol edilir
İndeks oluşturma parametreleri:

dimension=384: Vektör boyutu (all-MiniLM-L6-v2 modelinin çıkış boyutu)
metric="dotproduct": Vektörler arası benzerlik ölçümü
ServerlessSpec: AWS'de serverless olarak çalışacak
region="us-east-1": Veri merkezi konumu
```

In [36]:
import os
from pinecone import Pinecone,ServerlessSpec
index_name = "hybrid-search-langchain-pinecone"

# initialize the pinecone client
pc = Pinecone(api_key=api_key)

# create the index
if index_name not in pc.list_indexes().names():
    pc.create_index(name=index_name, dimension=384, metric="dotproduct",spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    ))

```markdown
Adım 5: İndeks Bağlantısı
Oluşturulan veya mevcut indekse bağlantı kurar
```

In [37]:
index = pc.Index(index_name)
index

```markdown
Adım 6: HuggingFace Embeddings
all-MiniLM-L6-v2: Hızlı ve etkili sentence embedding modeli
384 boyutlu vektörler üretir
HuggingFace token: Modeli indirmek için gerekli (rate limit önlemi)
```

In [38]:
# vector embedding and sparse matrix
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

```markdown
Adım 7: BM25 Encoder
BM25 Nedir:

Sözlüksel arama algoritması
Term frequency ve inverse document frequency kullanır
Exact keyword match için mükemmel
Semantik embedding ile birleştiğinde çok güçlü sonuçlar verir
```

In [39]:
from pinecone_text.sparse import BM25Encoder
bm25Encoder = BM25Encoder().default()
bm25Encoder

```markdown
Adım 8: BM25 Training
Örnek cümleler tanımlanır
BM25 modeli bu cümlelerle eğitilir
IDF değerleri hesaplanır ve JSON dosyasına kaydedilir
Bu değerler gelecekte yeniden yüklenmek için saklanır
```

In [40]:
sentences = [
    "In 2025, I visited Paris.",
    "In 2024, I visited London.",
    "In 2023, I visited New York."
]

# tf-idf values on this sentences
bm25Encoder.fit(sentences)

# store the values to a json file 
bm25Encoder.dump("bm25_values.json")

100%|██████████| 3/3 [00:00<00:00, 2979.61it/s]


```markdown
Hibrit Retriever Kurulumu
Adım 9: Retriever Oluşturma
Hibrit Arama Avantajları:

Semantik arama: "Paris" ile "Fransa'nın başkenti" arasında bağlantı kurar
Sözlüksel arama: Exact kelimeleri bulur ("2024" aramasında tam eşleşme)
Birleşik sonuç: Her iki yöntemin güçlü yanlarını kullanır
```

In [41]:
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25Encoder,
    index=index
)
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x0000024F388B4FE0>, index=<pinecone.db_data.index.Index object at 0x0000024F38D5BCE0>)

```markdown
Adım 10: Veri Ekleme
Önceden tanımlanan cümleler indekse eklenir
Hem vektör hem de sparse indeksleme yapılır
```

In [42]:
retriever.add_texts(sentences)

100%|██████████| 1/1 [00:00<00:00,  1.09it/s]


```markdown
Adım 11: Query Testi
```

In [43]:
retriever.invoke("Which city did I visit first?")

[Document(metadata={'score': 0.257063448}, page_content='In 2024, I visited London.'),
 Document(metadata={'score': 0.245632201}, page_content='In 2023, I visited to London.'),
 Document(metadata={'score': 0.240455672}, page_content='In 2023, I visited New York.'),
 Document(metadata={'score': 0.230221242}, page_content='In 2023, I visited Paris.')]

```markdown
Sistemin Çalışma Mantığı
Hibrit Arama Süreci:

Query Processing: "Which city did I visit first?" sorusu işlenir
Vector Search: Embedding modeli ile semantik benzerlik bulunur
Sparse Search: BM25 ile keyword-based arama yapılır
Score Fusion: Her iki skorun kombinasyonu hesaplanır
Ranking: Sonuçlar birleşik skora göre sıralanır

Sonuç Analizi:

En yüksek skor: "In 2024, I visited London." (0.257)
İkinci: "In 2023, I visited to London." (0.245)

Bu cümlede "to" eklenmesi ilginç (muhtemelen veri girişi hatası)


Sistem semantik anlam ve keyword eşleşmesi kombinasyonunu kullanarak sıralama yapmış
```

```markdown
Kullanım Alanları
Bu sistem şu durumlarda çok etkilidir:

Dokuman arama: Hem anlam hem de exact keyword'lerle
FAQ sistemleri: Kullanıcı sorularını doğru cevaplarla eşleştirme
E-ticaret arama: Ürün ismi ve özellikleri kombinasyonu
Hukuki dokuman arama: Exact terimler + konsept araması

Bu hibrit yaklaşım, tek başına vektör veya keyword aramadan çok daha güçlü ve esnek sonuçlar sağlar.
```

```markdown
neden pinecone kullanıyoruz burada
Edit

Pinecone'un bu projede kullanılmasının teknik ve pratik sebeplerini açıklayayım:
1. Vektör Veritabanı İhtiyacı
python

# Klasik veritabanları şunu yapabilir:
SELECT * FROM documents WHERE title = 'Paris'

# Ama şunu yapamaz:
"Paris ile semantik olarak benzer şehirleri bul"
# (Roma, Londra, Barcelona gibi)

Vektör veritabanları bu semantik aramaları mümkün kılar. Pinecone özellikle bu iş için optimize edilmiş.
2. Neden Başka Alternatif Değil?
Geleneksel Veritabanları (MySQL, PostgreSQL):
python

# ❌ Vektör araması için optimize değil
# ❌ 384 boyutlu vektörlerle cosine similarity hesabı çok yavaş
# ❌ Milyonlarca vektör için indexleme zor

Elasticsearch:
python

# ✅ Text search'te güçlü
# ❌ Vektör araması için ek plugin gerekiyor
# ❌ Kurulum ve yönetim karmaşık
# ❌ Hibrit arama (vektör + BM25) native değil

Diğer Vektör DB'leri (Weaviate, Chroma, Qdrant):
python

# ✅ Vektör araması yapabilir
# ❌ Pinecone kadar mature değil
# ❌ Serverless seçenekleri sınırlı
# ❌ LangChain entegrasyonu daha az gelişmiş

3. Pinecone'un Benzersiz Avantajları
A) Hibrit Arama Native Desteği:
python

# Pinecone tek sorguda hem vektör hem sparse arama yapar
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,      # Semantik arama
    sparse_encoder=bm25Encoder, # Keyword arama  
    index=index                 # İkisi birden!
)

Diğer sistemlerde bu çok daha karmaşık:
python

# İki ayrı sistem kurmanız gerekir:
# 1. Elasticsearch (BM25 için)
# 2. Ayrı vektör DB (semantik arama için)
# Sonra skorları manuel birleştirmeniz gerekir

B) Serverless Mimari:
python

spec=ServerlessSpec(
    cloud="aws",
    region="us-east-1"
)

Bu şu demek:

    Otomatik ölçeklendirme: Trafik artınca sistem büyür
    Sıfır maintenance: Server yönetimi yok
    Pay-per-use: Sadece kullandığınız kadar öde
    Global CDN: Dünya çapında hızlı erişim

C) Production-Ready Özellikler:
python

# ✅ Automatic backups
# ✅ Multi-region replication  
# ✅ Role-based access control
# ✅ Monitoring ve alerting
# ✅ 99.9% uptime SLA

4. Bu Projede Neden Kritik?
Veri Türü:
python

sentences = [
    "In 2025, I visited Paris.",
    "In 2024, I visited London.", 
    "In 2023, I visited New York."
]

Bu seyahat verileri için:

    "Which city did I visit first?" sorusu semantik anlama ihtiyaç duyar
    "2024" araması exact keyword match istir
    İkisini birden yapabilen sistem gerek

Skorlama Sistemi:
python

# Pinecone hibrit skorları otomatik birleştirir:
Document(metadata={'score': 0.257063448}, page_content='In 2024, I visited London.')

Manuel yapılsaydı:
python

# Çok karmaşık olurdu:
semantic_score = cosine_similarity(query_vector, doc_vector)  
bm25_score = calculate_bm25(query_terms, document)
final_score = alpha * semantic_score + beta * bm25_score  # Alpha/beta nasıl ayarlanır?

5. Alternatif Yaklaşımların Sorunları
Sadece Semantic Search:
python

query = "Which city did I visit first?"
# ❌ "first" kelimesi tarihsel sıralamayı anlayamayabilir
# ❌ "2023, 2024, 2025" numerical pattern'ini kaçırabilir

Sadece Keyword Search:
python

query = "Which city did I visit first?"
# ❌ "first" kelimesi dokumanlarda geçmiyor
# ❌ "earliest", "initially" gibi sinonimlerle eşleşemez

Hibrit Pinecone:
python

# ✅ "first" → "earliest date" semantic anlama
# ✅ "2023" → exact year match
# ✅ İkisini birleştirerek optimal sonuç

6. Maliyet-Performans Analizi
Self-hosted Alternatifi:
python

# Elasticsearch + Custom Vector Search:
# - EC2 instance: $100/month
# - Maintenance time: 10 saat/ay
# - Scaling complexity: Yüksek
# - Downtime riski: Var

Pinecone:
python

# - Sadece kullanım ücreti
# - Zero maintenance
# - Automatic scaling
# - 99.9% uptime guarantee
# Küçük-orta projeler için çok daha ekonomik

7. LangChain Ekosistemi
python

from langchain_community.retrievers import PineconeHybridSearchRetriever

Bu satır çok önemli çünkü:

    Tek satırda hibrit arama kurulumu
    Standardize interface: Başka RAG bileşenleriyle uyumlu
    Battle-tested: Binlerce production deployment'ta kullanılıyor

Sonuç: Neden Pinecone?

    Tek platformda hibrit arama (vektör + BM25)
    Zero-ops serverless mimari
    Production-ready stability
    LangChain native entegrasyon
    Cost-effective küçük-orta projeler için
    Global scale performance

Bu proje için Pinecone, "right tool for the job" seçimi. Hem teknik gereksinimleri karşılıyor hem de operasyonel yükü minimuma indiriyor.



```